# Análise Exploratória de Dados (EDA) - Mercado de Veículos EUA
Este notebook documenta a análise inicial do conjunto de dados `vehicles.csv`. 
O objetivo é validar a qualidade dos dados e testar as visualizações que serão implementadas no dashboard Streamlit.

In [ ]:
# !pip install pandas plotly import sys#
#import sys
# python = sys.executable
# !$python -m pip install pandas plotly nbformat 
# %pip install pandas plotly nbformat 


In [10]:
import pandas as pd
import plotly.express as px
from pathlib import Path

# Define o caminho para a pasta onde o notebook está
current_dir = Path.cwd()

# Lista de nomes possíveis para o arquivo (na pasta raiz, um nível acima)
# Tentamos encontrar na pasta pai (..)
candidate_files = [
    current_dir.parent / "vehicles_us.csv",
    current_dir.parent / "vehicles.csv",
    current_dir.parent / "vehicles_us.csv"
]

# Procura qual deles existe
data_path = next((p for p in candidate_files if p.exists()), None)

if data_path:
    df = pd.read_csv(data_path)
    print(f"Sucesso! Arquivo encontrado em: {data_path}")
    print(f"O dataset possui {df.shape[0]} linhas e {df.shape[1]} colunas.")
    display(df.head())
else:
    print("ERRO: O arquivo CSV não foi encontrado na pasta raiz.")
    print(f"Arquivos na pasta raiz: {[f.name for f in current_dir.parent.iterdir()]}")

Sucesso! Arquivo encontrado em: c:\Users\rogerio.barberi\OneDrive - Dracones Consultoria em TI\Pessoal\Triple Ten\Cinconotebook\vehicles_env\vehicles_us.csv
O dataset possui 51525 linhas e 13 colunas.


,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
0,9400,2011.0,bmw x5,good,6.0,gas,145000.0,automatic,SUV,NaN,1.0,2018-06-23,19
1,25500,NaN,ford f-150,good,6.0,gas,88705.0,automatic,pickup,white,1.0,2018-10-19,50
2,5500,2013.0,hyundai sonata,like new,4.0,gas,110000.0,automatic,sedan,red,NaN,2019-02-07,79
3,1500,2003.0,ford f-150,fair,8.0,gas,NaN,automatic,pickup,NaN,NaN,2019-03-22,9
4,14900,2017.0,chrysler 200,excellent,4.0,gas,80903.0,automatic,sedan,black,NaN,2019-04-02,28


In [11]:
# Verificando valores ausentes e tipos de dados
print(df.info())
print("\nValores nulos por coluna:")
print(df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  str    
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   42258 non-null  str    
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 5.1 MB
None

Valores nulos por coluna:
price               0
model_year       3619
model               0
condition           0
cylinders        5260
fuel                0
odometer         7892
transmission 

### Tratamento de Dados Ausentes
Para evitar a perda de 20% do dataset, utilizaremos a **Imputação por Mediana de Grupo**:
- Ano do modelo e Cilindros serão preenchidos pela mediana do **modelo**.
- Quilometragem (odômetro) será preenchida pela mediana do **ano do modelo**.

In [12]:
# 1. Imputação inteligente
df['model_year'] = df['model_year'].fillna(df.groupby('model')['model_year'].transform('median'))
df['cylinders'] = df['cylinders'].fillna(df.groupby('model')['cylinders'].transform('median'))
df['odometer'] = df['odometer'].fillna(df.groupby('model_year')['odometer'].transform('median'))

# 2. Padronização de booleanos
df['is_4wd'] = df['is_4wd'].fillna(0)

# 3. Criação da coluna Brand (Marca)
df['brand'] = df['model'].apply(lambda x: str(x).split()[0])

# Removendo nulos restantes que não puderam ser imputados
df.dropna(subset=['price', 'model_year', 'odometer'], inplace=True)

print("Nulos restantes:", df.isna().sum().sum())

Nulos restantes: 9267


In [13]:
# Testando o histograma que irá para o App
fig = px.histogram(df, x="price", title="Distribuição de Preços - Teste EDA")
fig.show()

# Testando o gráfico de dispersão
fig_scatter = px.scatter(df, x="odometer", y="price", opacity=0.3, title="Preço vs Quilometragem")
fig_scatter.show()